# Iron Plates

**Type:** Producer — stores output in dimensional depot  
**Blueprint:** `iron-plate` (120 Iron Ingots/min → 80 Iron Plates/min at 100%)  
**Downstream:** Reinforced Iron Plates pulls from this storage

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path('..').resolve()))

from blueprints import BLUEPRINTS, Blueprint, Stage
from satisfactory import ITEMS
import pandas as pd

BP  = BLUEPRINTS
TOL = 0.5


def machine_hover(bp: Blueprint, output_key: str, target_rate: float) -> str:
    if not bp.stages:
        return ""
    rates = bp.stage_rates(output_key, target_rate)
    lines = ["── Machines (per copy) ──"]
    for r in rates:
        name = ITEMS[r["item_key"]].name
        lines.append(f"  {r['machines']}× {r['building'].title()} → {r['per_machine_rate']:.2f} {name}/min each")
    return "<br>" + "<br>".join(lines)

## Constants

Set `IRON_INGOTS` to your allocation from the 270/min supply.  
Set `STASH_RATE` to how many Iron Plates/min should accumulate in storage — the Reinforced Iron Plates notebook pulls the remainder.

In [ ]:
# ── Supply ────────────────────────────────────────────────────────────────────
IRON_INGOTS = None  # ← your ingot allocation (items/min)

# ── Stash target ──────────────────────────────────────────────────────────────
STASH_RATE = 20.0   # Iron Plates/min accumulating in storage

# ── Blueprint placement ───────────────────────────────────────────────────────
PLATE_COPIES      = None  # ← number of blueprints to place
PLATE_OUTPUT_EACH = None  # ← output rate per copy to set in-game (Iron Plates/min)

## Derived rates and balance

In [ ]:
assert None not in (IRON_INGOTS, PLATE_COPIES, PLATE_OUTPUT_EACH), \
    "Fill in all constants in the cell above before running this cell."

plate_total  = PLATE_COPIES * PLATE_OUTPUT_EACH
plate_ingots = plate_total  * BP['iron-plate'].input_ratio('iron-ingot', 'iron-plate')

available_for_reinforced = plate_total - STASH_RATE

assert abs(plate_ingots - IRON_INGOTS) < TOL, (
    f"Ingot balance: consuming {plate_ingots:.2f}/min, supplied {IRON_INGOTS:.0f}/min "
    f"(delta {plate_ingots - IRON_INGOTS:+.2f})"
)
assert available_for_reinforced > 0, (
    f"Not enough plates to stash {STASH_RATE}/min — only producing {plate_total:.2f}/min"
)

print(f"✓ Chain balanced")
print(f"  {IRON_INGOTS:.0f} Iron Ingots/min  →  {plate_total:.2f} Iron Plates/min")
print(f"  Stashing:                          {STASH_RATE:.0f} Iron Plates/min")
print(f"  Available for Reinforced:          {available_for_reinforced:.2f} Iron Plates/min")